**Continue fine-tuning fastText LID-176 on your 11 groups**

Run cells in order. This imports both pretrained matrices, verifies the original model, adds Pali, and trains all inherited parameters. Outputs remain available for all 177 languages. Read `FASTTEXT_CONTINUATION.md` for design details.

Before opening this notebook: copy the package files into your repo, push them to your experiment branch, and copy your **three new dataset files** into the Drive folder below. Select CPU for the pilot or a GPU runtime for the full PyTorch run. The full run starts fresh from `init177`, not from the pilot.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git"
BRANCH = "experiment/fasttext-continuation"  # Must contain the new code files
PROJECT_DIR = Path("/content/langid")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/langid_finetuning_data")
DRIVE_EXPERIMENT_DIR = Path("/content/drive/MyDrive/fasttext_continuation_run01")

# First baseline uses the existing class distribution. For a separate weighted
# experiment, set BALANCE="group" and use a different FULL_RUN_NAME.
BALANCE = "none"
FULL_RUN_NAME = "full_none"
EPOCHS = 5
LEARNING_RATE = 0.01
BATCH_SIZE = 64
SEED = 42

**1. Load your code and dependencies.** If Colab already contains a different checkout at `/content/langid`, use a fresh runtime or a different `PROJECT_DIR`. This cell preserves existing checkout changes.

In [ ]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
requirements = PROJECT_DIR / "data_pipeline/fasttext_continual/requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError("The selected branch does not contain the new continuation files. Copy, commit and push them first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

**2. Mount Drive and copy the prepared data into the runtime.** Generated datasets are git-ignored and are not downloaded by cloning your repo. Keep checkpoints on Drive so they survive runtime disconnection.

In [ ]:
from google.colab import drive
import shutil
import json

drive.mount("/content/drive")
DATA_DIR = PROJECT_DIR / "data_pipeline/datasets/finetuning"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
filenames = ["train_mixed_11groups.jsonl", "val_mixed_11groups.jsonl", "dataset_11groups_report.json"]
for name in filenames:
    source = DRIVE_DATA_DIR / name
    if not source.is_file():
        raise FileNotFoundError(f"Copy your prepared file to Drive first: {source}")
    shutil.copy2(source, DATA_DIR / name)
    print(name, "copied")
TRAIN = DATA_DIR / filenames[0]
VAL = DATA_DIR / filenames[1]
shutil.copy2(DATA_DIR / filenames[2], DRIVE_EXPERIMENT_DIR / filenames[2])

**3. Download the original full checkpoint.** The official weights are distributed under [CC-BY-SA 3.0](https://fasttext.cc/docs/en/language-identification.html). The checksum below identifies the original binary used to verify this package.

In [ ]:
import urllib.request
from data_pipeline.fasttext_continual.model import BASE_URL, sha256

BASE_BIN = PROJECT_DIR / "data_pipeline/models/benchmark/fastText/lid.176.bin"
EXPECTED_SHA256 = "7e69ec5451bc261cc7844e49e4792a85d7f09c06789ec800fc4a44aec362764e"
BASE_BIN.parent.mkdir(parents=True, exist_ok=True)
if not BASE_BIN.exists():
    temporary = BASE_BIN.with_suffix(".bin.part")
    with urllib.request.urlopen(BASE_URL, timeout=60) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if sha256(temporary) != EXPECTED_SHA256:
        raise RuntimeError("Downloaded checkpoint checksum differs. Investigate; do not train.")
    temporary.replace(BASE_BIN)
if sha256(BASE_BIN) != EXPECTED_SHA256:
    raise RuntimeError("Existing lid.176.bin is not the original verified checkpoint.")
print("Original checkpoint SHA-256 verified.")

**4. Verify import, expand to 177 languages, and check gradients.** This runs on CPU. Every assertion must pass. It saves `base176/` and the untrained `init177/`. With a zero new node, the old Sinhala probability is initially divided equally between Sinhala and Pali.

In [ ]:
def run_module(module, *arguments):
    subprocess.run([sys.executable, "-m", "data_pipeline.fasttext_continual." + module,
                    *map(str, arguments)], cwd=PROJECT_DIR, check=True)

INIT_ROOT = DRIVE_EXPERIMENT_DIR / "initialization"
PARITY_REPORT = INIT_ROOT / "parity_report.json"
if not PARITY_REPORT.exists():
    run_module("verify", "--base-bin", BASE_BIN, "--data", TRAIN, "--data", VAL,
               "--out", INIT_ROOT)
report = json.loads(PARITY_REPORT.read_text())
assert report["passed"] and report["expansion_invariants_passed"]
assert report["checkpoint_roundtrip_passed"] and report["expanded_labels"] == 177
assert report["base_sha256"] == EXPECTED_SHA256
print(json.dumps(report, indent=2))
INIT_MODEL = INIT_ROOT / "init177"

**5. Record the original and expanded validation baselines.** Both Sanskrit groups use the language label `sa`. These checks use validation, not the benchmark test sets.

In [ ]:
run_module("evaluate", "--base-bin", BASE_BIN, "--data", VAL,
           "--out", INIT_ROOT / "native_validation.json")
run_module("evaluate", "--checkpoint", INIT_MODEL, "--data", VAL,
           "--out", INIT_ROOT / "expanded_validation.json")

**6. Run a small CPU pilot.** Up to 64 training and 16 validation examples per group, one epoch. This checks execution; its scores are not research results.

In [ ]:
PILOT_DIR = DRIVE_EXPERIMENT_DIR / "pilot"
pilot_args = ["--init", INIT_MODEL, "--train", TRAIN, "--val", VAL, "--out", PILOT_DIR,
              "--epochs", 1, "--lr", 0.01, "--batch-size", 64, "--device", "cpu",
              "--seed", SEED, "--pilot-per-group", 64, "--pilot-val-per-group", 16]
if (PILOT_DIR / "last.json").exists():
    pilot_args.append("--resume")
run_module("train", *pilot_args)

**7. Train on the complete prepared dataset.** This starts from the verified untrained `init177`. Checkpoints are saved after each epoch. If interrupted, rerun this cell with the same settings: it resumes from the last complete epoch. A different configuration needs a new run folder.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
FULL_RUN = DRIVE_EXPERIMENT_DIR / FULL_RUN_NAME
full_args = ["--init", INIT_MODEL, "--train", TRAIN, "--val", VAL, "--out", FULL_RUN,
             "--epochs", EPOCHS, "--lr", LEARNING_RATE, "--batch-size", BATCH_SIZE,
             "--balance", BALANCE, "--seed", SEED, "--device", DEVICE]
if (FULL_RUN / "last.json").exists():
    full_args.append("--resume")
run_module("train", *full_args)

**8. Compare validation results.** Macro-F1 is over the 10 distinct gold languages. The 11 group metrics are accuracies (recalls within each script subset), not 11-class F1. All 176/177 output labels compete at inference. German has only 18 validation examples, so its estimate is especially variable.

In [ ]:
BEST = FULL_RUN / "best"
run_module("evaluate", "--checkpoint", BEST, "--data", VAL,
           "--out", FULL_RUN / "final_validation.json", "--device", DEVICE)
for title, path in [
    ("Original 176", INIT_ROOT / "native_validation.json"),
    ("Expanded 177, untrained", INIT_ROOT / "expanded_validation.json"),
    ("Continued 177, best trained epoch", FULL_RUN / "final_validation.json"),
]:
    result = json.loads(path.read_text())
    print(f"{title:36s} macro-F1={result['language_macro_f1']:.6f} accuracy={result['accuracy']:.6f}")
print("Epoch history:")
print((FULL_RUN / "history.json").read_text())

**9. Load your complete checkpoint for predictions.** Keep `weights.pt`, `config.json` and `vocab.json` together. The expanded hierarchy requires this loader; it is not a native fastText `.bin`.

In [ ]:
from data_pipeline.fasttext_continual import ContinualLID

model = ContinualLID.from_pretrained(BEST, device="cpu")
labels, scores = model.predict("සිංහල භාෂාව ශ්‍රී ලංකාවේ භාවිතා වේ.", k=3)
print(list(zip(labels, scores)))

**10. Optional final benchmark evaluation — run after freezing model selection.**

This compares the original model, the untrained expansion and your best trained model on the repository's three 11-group test CSVs. Verify that these are the intended held-out files covered by your dataset overlap checks. Their language–script labels are mapped consistently. Do not tune hyperparameters using these test results.

These 11 groups alone do not establish retention across all original languages. Broader retention evaluation is a later step.

In [ ]:
BENCHMARK_FILES = [
    PROJECT_DIR / "data/phase2_eval/eval_flores_plus_11lang.csv",
    PROJECT_DIR / "data/phase2_eval/eval_wili_2018_11lang.csv",
    PROJECT_DIR / "data/phase2_eval/eval_commonlid_11lang.csv",
]
missing = [str(p) for p in BENCHMARK_FILES if not p.is_file()]
if missing:
    raise FileNotFoundError("Provide the intended benchmark files: " + ", ".join(missing))
RESULTS_DIR = FULL_RUN / "final_benchmarks"
for dataset in BENCHMARK_FILES:
    for name, flag, checkpoint in [
        ("original176", "--base-bin", BASE_BIN),
        ("expanded177", "--checkpoint", INIT_MODEL),
        ("continued177", "--checkpoint", BEST),
    ]:
        run_module("evaluate", flag, checkpoint, "--data", dataset,
                   "--out", RESULTS_DIR / f"{dataset.stem}_{name}.json")